In [ ]:
# 1) Install + Imports
!pip -q install tensorflow tensorflow_hub matplotlib pillow

import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# 2) Helpers: load + preprocess images
def load_image(path, max_dim=512):
    img = Image.open(path).convert("RGB")

    # Resize while keeping aspect ratio
    scale = max_dim / max(img.size)
    new_size = (int(img.size[0] * scale), int(img.size[1] * scale))
    img = img.resize(new_size, Image.LANCZOS)

    img = np.array(img).astype(np.float32) / 255.0
    img = img[None, ...]  # add batch dim
    return tf.convert_to_tensor(img)

def show(img, title=None):
    img = tf.squeeze(img)
    plt.figure(figsize=(7, 7))
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()

def save_image(tensor, out_path="stylized.png"):
    img = tf.squeeze(tensor).numpy()
    img = (img * 255).clip(0, 255).astype(np.uint8)
    Image.fromarray(img).save(out_path)
    return out_path

In [ ]:
# 3) Load Content + Style Images
# ------------------------------------------------------------
# Option 1 (DEFAULT): Download sample images (no dataset needed)
!wget -q -O content.jpg https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg
!wget -q -O style.jpg https://storage.googleapis.com/download.tensorflow.org/example_images/Vassily_Kandinsky%2C_1913_-_Composition_7.jpg
print("Downloaded sample images: content.jpg and style.jpg")

# Option 2 (Optional): Upload your own images (auto-detect filenames)
# ------------------------------------------------------------
# Uncomment to upload TWO images. First selected = content, second = style.
# This avoids renaming files.
#
# from google.colab import files
# uploaded = files.upload()
# uploaded_names = list(uploaded.keys())
# assert len(uploaded_names) >= 2, "Upload at least TWO images (content + style)."
# CONTENT_PATH = uploaded_names[0]
# STYLE_PATH   = uploaded_names[1]
# print("Using uploaded images:")
# print("  Content:", CONTENT_PATH)
# print("  Style  :", STYLE_PATH)

# If no upload happened, use the downloaded defaults
if "CONTENT_PATH" not in globals():
    CONTENT_PATH = "content.jpg"
if "STYLE_PATH" not in globals():
    STYLE_PATH = "style.jpg"

content = load_image(CONTENT_PATH)
style   = load_image(STYLE_PATH)

show(content, "Content Image")
show(style, "Style Image")

In [ ]:
# 4) Run Style Transfer
hub_model = hub.load("https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2")
stylized = hub_model(content, style)[0]

show(stylized, "Stylized Output")

In [ ]:
# 5) Save output
out = save_image(stylized, "stylized_output.png")
print("Saved:", out)

from google.colab import files
files.download(out)